<div style="text-align:center"><strong style="color:darkred;font-size:38px">The Orthogonal Procrustes Problem</strong></div>

# 1. Aligning Measurements with a Reference

An engineering team measures the same calibration targets in two coordinate frames. Their design coordinates are known, while a sensor reports coordinates in its own frame. The sensor frame may be rotated and translated relative to the design frame, and noise prevents exact agreement.

The **orthogonal Procrustes problem** finds the orthogonal transformation that best aligns corresponding point sets in the least-squares sense. Its solution is a direct application of the singular value decomposition (SVD).

This notebook derives and implements the solution for a planar sensor-calibration example. It distinguishes a general orthogonal transformation, which may include reflection, from a proper rotation, which must preserve orientation.

### Learning objectives

- Formulate point-set alignment as a Frobenius-norm minimization.
- Remove translation by centering the points.
- Derive the optimal orthogonal transformation using an SVD.
- Enforce determinant one when only physical rotations are admissible.
- Assess alignment quality and recognize poorly identified fits.

Prerequisites: matrix transposes, Frobenius norm, trace, orthogonal matrices, and SVD. See [27_SVD.ipynb](27_SVD.ipynb) and [29_SVDapplications.ipynb](29_SVDapplications.ipynb).


# 2. Data Convention and Objective

Let $X,Y\in\mathbb{R}^{d\times n}$ contain $n$ corresponding points as **columns**. The $j$th columns $x_j$ and $y_j$ represent the same physical target in two coordinate systems. We seek an orthogonal matrix $Q$ and translation $t$ such that

$\qquad Y\approx QX+t\mathbf{1}^T,\qquad Q^TQ=I.$

The Procrustes objective sums squared coordinate errors over all correspondences:

$\qquad \underset{Q^TQ=I}{\operatorname{minimize}}\;\|QX-Y\|_F^2.$

Orthogonality preserves lengths and angles; it excludes shear and nonuniform scaling. Some references store points as rows and write $\min_R\|XR-Y\|_F$. That is the same problem after transposing the data and setting $R=Q^T$. Keeping the convention explicit prevents implementation errors.

## 2.1 Eliminate Translation

For fixed $Q$, the least-squares translation is the difference of the sample means:

$\qquad \bar{x}=\frac1nX\mathbf{1},\qquad \bar{y}=\frac1nY\mathbf{1},\qquad t=\bar{y}-Q\bar{x}.$

Define $X_c=X-\bar{x}\mathbf{1}^T$ and $Y_c=Y-\bar{y}\mathbf{1}^T$. Substituting the optimal translation reduces the problem to

$\qquad \underset{Q^TQ=I}{\operatorname{minimize}}\;\|QX_c-Y_c\|_F^2.$

Thus we estimate the rotation/reflection from centered geometry, then recover translation from the centroids.


# 3. Deriving the SVD Solution

Using $\|M\|_F^2=\operatorname{tr}(M^TM)$ and $Q^TQ=I$,

$\qquad \begin{aligned}
\|QX_c-Y_c\|_F^2
&=\operatorname{tr}(X_c^TX_c)+\operatorname{tr}(Y_c^TY_c)
-2\operatorname{tr}(Q^TY_cX_c^T).
\end{aligned}$

Only the final term depends on $Q$. Let $C=Y_cX_c^T$; minimizing the residual is equivalent to maximizing $\operatorname{tr}(Q^TC)$. Compute

$\qquad C=U\Sigma V^T.$

Set $Z=U^TQV$. This is orthogonal, and cyclic invariance of trace gives

$\qquad \operatorname{tr}(Q^TC)=\operatorname{tr}(Z^T\Sigma)\leq\sum_i\sigma_i.$

The bound is attained at $Z=I$, so the optimal orthogonal matrix is

$\qquad Q_*=UV^T.$

The SVD therefore solves the constrained least-squares problem exactly, rather than approximately.


In [ ]:
import numpy as np
import holoviews as hv
import panel as pn
import param

hv.extension("bokeh")
pn.extension()
np.set_printoptions(precision=4, suppress=True)
print("NumPy:", np.__version__)
print("HoloViews:", hv.__version__)
print("Panel:", pn.__version__)

In [ ]:
def center_points(points):
    """Center column-point data and return the centroid."""
    points = np.asarray(points, dtype=float)
    if points.ndim != 2 or points.shape[1] == 0:
        raise ValueError("points must have shape (dimension, number_of_points)")
    centroid = points.mean(axis=1, keepdims=True)
    return points - centroid, centroid


def orthogonal_procrustes(X, Y):
    """Minimize ||Q Xc - Yc||_F over orthogonal Q; reflections are allowed."""
    X, Y = np.asarray(X, dtype=float), np.asarray(Y, dtype=float)
    if X.ndim != 2 or Y.ndim != 2 or X.shape != Y.shape:
        raise ValueError("X and Y must have equal (dimension, points) shapes")
    Xc, xbar = center_points(X)
    Yc, ybar = center_points(Y)
    U, singular_values, Vt = np.linalg.svd(Yc @ Xc.T)
    Q = U @ Vt
    t = (ybar - Q @ xbar).ravel()
    return Q, t, singular_values


def proper_rotation_procrustes(X, Y):
    """Fit Y ~= Q X + t with Q orthogonal and det(Q)=+1."""
    X, Y = np.asarray(X, dtype=float), np.asarray(Y, dtype=float)
    if X.ndim != 2 or Y.ndim != 2 or X.shape != Y.shape:
        raise ValueError("X and Y must have equal (dimension, points) shapes")
    Xc, xbar = center_points(X)
    Yc, ybar = center_points(Y)
    U, singular_values, Vt = np.linalg.svd(Yc @ Xc.T)
    D = np.eye(X.shape[0])
    D[-1, -1] = np.linalg.det(U @ Vt)
    Q = U @ D @ Vt
    t = (ybar - Q @ xbar).ravel()
    return Q, t, singular_values


def alignment_residuals(Q, t, X, Y):
    """Return fitted points and one Euclidean residual per correspondence."""
    fitted = Q @ X + np.asarray(t).reshape(-1, 1)
    return fitted, np.linalg.norm(fitted - Y, axis=0)

## 3.1 Verify the Implementation

On exact data, the fitted transform should reproduce the reference coordinates up to roundoff. This checks centering, cross-covariance orientation, and translation recovery together.


In [ ]:
X_check = np.array([[-1.0, 0.0, 1.0, 0.5],
                     [-0.5, 1.0, -0.5, 0.8]])
angle = np.deg2rad(27.0)
Q_check = np.array([[np.cos(angle), -np.sin(angle)],
                    [np.sin(angle),  np.cos(angle)]])
t_check = np.array([2.0, -1.5])
Y_check = Q_check @ X_check + t_check[:, None]
Q_fit, t_fit, sigma = proper_rotation_procrustes(X_check, Y_check)
Y_fit, residuals = alignment_residuals(Q_fit, t_fit, X_check, Y_check)
print("maximum point error:", residuals.max())
print("rotation error:", np.linalg.norm(Q_fit - Q_check))
print("translation estimate:", t_fit)
print("det(Q):", np.linalg.det(Q_fit))
assert np.allclose(Y_fit, Y_check)
assert np.allclose(Q_fit.T @ Q_fit, np.eye(2))
assert np.linalg.det(Q_fit) > 0

# 4. Orthogonal Map or Physical Rotation?

An orthogonal matrix has determinant $+1$ or $-1$. Determinant $+1$ represents a proper rotation; determinant $-1$ includes a reflection. The unconstrained solution $UV^T$ can be a reflection.

If the application requires a physical rotation, add the constraint $\det(Q)=1$. With $C=U\Sigma V^T$, define $D=\operatorname{diag}(1,\ldots,1,\det(UV^T))$ and use

$\qquad Q_{\mathrm{rot}}=UDV^T,\qquad \det(Q_{\mathrm{rot}})=1.$

When a correction is needed, the objective loses $2\sigma_{\min}$ relative to the unrestricted optimum. Do not force determinant one without deciding which transformations are physically admissible.


In [ ]:
# A reflected point set makes the unrestricted optimum improper.
X_reflect = np.array([[-1.0, 1.0, 1.0, -1.0, 0.2],
                      [-0.7, -0.8, 0.9, 0.6, 0.4]])
F = np.diag([-1.0, 1.0])
Y_reflect = F @ X_reflect + np.array([[0.3], [-0.2]])
Q_o, t_o, _ = orthogonal_procrustes(X_reflect, Y_reflect)
Q_r, t_r, _ = proper_rotation_procrustes(X_reflect, Y_reflect)
_, r_o = alignment_residuals(Q_o, t_o, X_reflect, Y_reflect)
_, r_r = alignment_residuals(Q_r, t_r, X_reflect, Y_reflect)
print("unrestricted det(Q):", np.linalg.det(Q_o))
print("proper-rotation det(Q):", np.linalg.det(Q_r))
print("unrestricted residual norm:", np.linalg.norm(r_o))
print("proper-rotation residual norm:", np.linalg.norm(r_r))
assert np.linalg.det(Q_o) < 0
assert np.isclose(np.linalg.det(Q_r), 1.0)

The larger proper-rotation residual is expected: the data were generated by a reflection. It can reveal a mirrored coordinate convention or incorrect correspondences.

### SciPy convention

SciPy's orthogonal Procrustes routine solves the row-oriented problem $\min_R\|AR-B\|_F$ and allows reflections. With our column-point convention, pass transposed, centered data and transpose the returned matrix back. This provides an independent check for the unrestricted solution.


In [ ]:
from scipy.linalg import orthogonal_procrustes as scipy_orthogonal_procrustes

Xc, _ = center_points(X_check)
Yc, _ = center_points(Y_check)
R_scipy, scipy_scale = scipy_orthogonal_procrustes(Xc.T, Yc.T)
Q_scipy = R_scipy.T
print("our / SciPy rotation difference:", np.linalg.norm(Q_scipy - Q_check))
print("SciPy scale (sum of singular values):", scipy_scale)
assert np.allclose(Q_scipy, Q_check)

# 5. Engineering Example: Planar Sensor Calibration

Suppose a planar inspection sensor observes fiducial targets whose design-frame coordinates are known. The sensor axes are rotated and translated relative to the design frame. We estimate that rigid transform from corresponding target locations.

Targets should be spread across the plane rather than nearly collinear. Fixed seeded noise makes the example reproducible; a slider will vary its amplitude.


In [ ]:
Y_design = np.array([
    [0.0, 1.2, 2.6, 0.4, 1.8, 2.9, 0.7, 2.2],
    [0.0, 0.2, 0.1, 1.0, 1.3, 1.1, 2.4, 2.6],
])
true_angle = np.deg2rad(34.0)
Q_true = np.array([[np.cos(true_angle), -np.sin(true_angle)],
                   [np.sin(true_angle),  np.cos(true_angle)]])
t_true = np.array([1.1, -0.65])
X_sensor_exact = Q_true.T @ (Y_design - t_true[:, None])
rng = np.random.default_rng(20261007)
standardized_noise = rng.normal(size=X_sensor_exact.shape)
print("number of targets:", X_sensor_exact.shape[1])
print("true rotation (degrees):", np.rad2deg(true_angle))
print("true translation:", t_true)

## 5.1 Explore the Fit as Noise Changes

The slider changes sensor-coordinate noise. The left plot compares raw sensor and design coordinates. The right plot shows transformed readings over the design targets; connector segments show residuals.

This is ordinary least squares: all correspondences have equal weight, and an incorrect point can influence the fit. Interpret the residual summary together with the plot.


In [ ]:
class CalibrationDemo(param.Parameterized):
    noise_sd = param.Number(default=0.035, bounds=(0.0, 0.18), step=0.005,
                            label="Sensor noise standard deviation")

    @param.depends("noise_sd")
    def view(self):
        X_sensor = X_sensor_exact + self.noise_sd * standardized_noise
        Q_hat, t_hat, _ = proper_rotation_procrustes(X_sensor, Y_design)
        Y_aligned, residuals = alignment_residuals(Q_hat, t_hat, X_sensor, Y_design)
        raw = hv.Points((X_sensor[0], X_sensor[1]), label="Sensor readings").opts(
            color="#D55E00", size=8, alpha=0.8)
        raw *= hv.Points((Y_design[0], Y_design[1]), label="Design targets").opts(
            color="#0072B2", size=8)
        links = hv.Path([
            [(Y_design[0, j], Y_design[1, j]),
             (Y_aligned[0, j], Y_aligned[1, j])]
            for j in range(Y_design.shape[1])
        ]).opts(color="#999999", line_width=1, alpha=0.7)
        aligned = links * hv.Points(
            (Y_aligned[0], Y_aligned[1]), label="Transformed readings"
        ).opts(color="#D55E00", size=8)
        aligned *= hv.Points(
            (Y_design[0], Y_design[1]), label="Design targets"
        ).opts(color="#0072B2", size=8, marker="cross")
        style = dict(width=420, height=360, aspect="equal",
                     xlabel="x coordinate", ylabel="y coordinate",
                     show_grid=True, legend_position="top_left", toolbar="above")
        raw, aligned = raw.opts(**style), aligned.opts(**style)
        angle_hat = np.rad2deg(np.arctan2(Q_hat[1, 0], Q_hat[0, 0]))
        summary = pn.pane.Markdown(
            f"Estimated rotation: **{angle_hat:.2f} deg**  \n"
            f"Estimated translation: **({t_hat[0]:.3f}, {t_hat[1]:.3f})**  \n"
            f"RMS point residual: **{np.sqrt(np.mean(residuals**2)):.4f}**  \n"
            f"Maximum residual: **{residuals.max():.4f}**  \n"
            f"det(Q): **{np.linalg.det(Q_hat):.6f}**")
        return pn.Column(summary, pn.Row(
            pn.pane.HoloViews(raw, sizing_mode="fixed"),
            pn.pane.HoloViews(aligned, sizing_mode="fixed")))


calibration_demo = CalibrationDemo()
pn.Column(calibration_demo.param.noise_sd,
          pn.panel(calibration_demo.view)).servable()

In the zero-noise limit the aligned points coincide up to roundoff. Increasing noise raises residuals and parameter error, even though the estimated transform remains the least-squares optimum. The singular values of $Y_cX_c^T$ indicate how strongly the point geometry constrains the fit.

## 5.2 Numerical Diagnostics

Check errors against the known transform, RMS and maximum residual, and the orthogonality defect $\|Q^TQ-I\|_F$. In a real calibration, use independent validation targets and report errors in physical units.


In [ ]:
noise_sd = calibration_demo.noise_sd
X_sensor = X_sensor_exact + noise_sd * standardized_noise
Q_hat, t_hat, singular_values = proper_rotation_procrustes(X_sensor, Y_design)
Y_aligned, residuals = alignment_residuals(Q_hat, t_hat, X_sensor, Y_design)
angle_hat = np.arctan2(Q_hat[1, 0], Q_hat[0, 0])

def wrap_angle(angle):
    return (angle + np.pi) % (2 * np.pi) - np.pi

print(f"rotation error (degrees): {np.rad2deg(wrap_angle(angle_hat - true_angle)):.4f}")
print("translation error:", np.linalg.norm(t_hat - t_true))
print("RMS residual:", np.sqrt(np.mean(residuals**2)))
print("maximum residual:", residuals.max())
print("orthogonality defect:", np.linalg.norm(Q_hat.T @ Q_hat - np.eye(2)))
print("cross-covariance singular values:", singular_values)
assert np.allclose(Q_hat.T @ Q_hat, np.eye(2), atol=1e-12)
assert np.isclose(np.linalg.det(Q_hat), 1.0)

# 6. Variants and Modeling Choices

## 6.1 Uniform Scale

If the coordinate systems differ by a common scale, use $Y\approx sQX+t\mathbf{1}^T$, $s\geq0$. For fixed admissible $Q$, the optimal scale is

$\qquad s_*=\max\left(0,\frac{\langle QX_c,Y_c\rangle_F}{\|X_c\|_F^2}\right).$

For the unrestricted fit, the numerator is $\sum_i\sigma_i$; for a proper rotation it is $\operatorname{tr}(D\Sigma)$. Include scale only when the measurement model supports it.

## 6.2 Unequal Measurement Precision

If target $j$ has confidence weight $w_j\geq0$, minimize $\sum_jw_j\|Qx_j+t-y_j\|_2^2$. The weighted centroids are $\bar{x}_w=Xw/(\mathbf{1}^Tw)$ and $\bar{y}_w=Yw/(\mathbf{1}^Tw)$. After weighted centering, use the SVD of $Y_cWX_c^T$, with $W=\operatorname{diag}(w)$.

## 6.3 More Than Two Point Sets

Two-set Procrustes has a closed-form SVD solution. **Generalized Procrustes analysis** aligns three or more sets to a common, unknown template and typically requires iteration. It is a distinct problem not solved here.


In [ ]:
def similarity_procrustes(X, Y, proper_rotation=True):
    """Fit Y ~= scale * Q @ X + t[:, None] with scale >= 0."""
    X, Y = np.asarray(X, dtype=float), np.asarray(Y, dtype=float)
    if X.ndim != 2 or Y.ndim != 2 or X.shape != Y.shape:
        raise ValueError("X and Y must have equal (dimension, points) shapes")
    Xc, xbar = center_points(X)
    Yc, ybar = center_points(Y)
    U, singular_values, Vt = np.linalg.svd(Yc @ Xc.T)
    D = np.eye(X.shape[0])
    if proper_rotation:
        D[-1, -1] = np.linalg.det(U @ Vt)
    Q = U @ D @ Vt
    scale = max(float(np.sum(np.diag(D) * singular_values) / np.sum(Xc**2)), 0.0)
    t = (ybar - scale * Q @ xbar).ravel()
    return scale, Q, t


scale_true = 1.7
Y_scaled = scale_true * Q_check @ X_check + t_check[:, None]
scale_fit, Q_scale_fit, t_scale_fit = similarity_procrustes(X_check, Y_scaled)
print("true / fitted scale:", scale_true, scale_fit)
print("reconstruction error:", np.linalg.norm(
    scale_fit * Q_scale_fit @ X_check + t_scale_fit[:, None] - Y_scaled))
assert np.isclose(scale_fit, scale_true)
assert np.allclose(scale_fit * Q_scale_fit @ X_check + t_scale_fit[:, None], Y_scaled)

# 7. Identifiability and Limitations

The algorithm returns an optimizer, but the data may not determine a reliable transform.

- **Insufficient geometric spread:** coincident or nearly collinear planar targets do not robustly constrain rotation. Small cross-covariance singular values indicate weakly informed directions.
- **Incorrect correspondences:** column $j$ in $X$ must match column $j$ in $Y$. Procrustes does not find correspondences.
- **Outliers:** ordinary least squares squares residuals and is sensitive to gross errors. Consider robust fitting, outlier rejection, or RANSAC when mismatches are plausible.
- **Wrong model:** orthogonal Procrustes cannot fit shear or nonuniform scale. Decide whether reflection or uniform scale is physically admissible.
- **Validation:** a small training residual does not guarantee accurate calibration. Evaluate independent targets.


In [ ]:
# One corrupted correspondence can influence the least-squares fit.
X_outlier = X_sensor.copy()
X_outlier[:, 3] += np.array([0.8, -0.6])
Q_out, t_out, _ = proper_rotation_procrustes(X_outlier, Y_design)
_, residuals_out = alignment_residuals(Q_out, t_out, X_outlier, Y_design)
angle_out = np.arctan2(Q_out[1, 0], Q_out[0, 0])
print("rotation change (degrees):",
      np.rad2deg(wrap_angle(angle_out - angle_hat)))
print("largest residual:", residuals_out.max())
print("RMS residual:", np.sqrt(np.mean(residuals_out**2)))

# 8. Summary

For corresponding points stored in columns, rigid alignment is $Y\approx QX+t\mathbf{1}^T$. Centering removes translation. The remaining objective reduces to maximizing a trace involving $C=Y_cX_c^T$. If $C=U\Sigma V^T$, the best orthogonal transformation is $Q=UV^T$; if only proper rotations are allowed, use $Q=UDV^T$ with determinant correction.

The SVD gives a closed-form fit and informative diagnostics. Procrustes assumes known correspondences and a rigid model; it does not handle mismatches, outliers, or general affine deformation automatically.

### Further reading

- [MIT OpenCourseWare, Lecture 34: Distance Matrices and the Procrustes Problem](https://ocw.mit.edu/courses/18-065-matrix-methods-in-data-analysis-signal-processing-and-machine-learning-spring-2018/resources/lecture-34-distance-matrices-procrustes-problem-first-project/)
- [A Purely Algebraic Justification of the Kabsch-Umeyama Algorithm](https://arxiv.org/abs/1902.03138)
- [SciPy orthogonal_procrustes documentation](https://docs.scipy.org/doc/scipy/reference/generated/scipy.linalg.orthogonal_procrustes.html)
- [Robotics Systems: Calibration and the Orthogonal Procrustes Problem](https://motion.cs.illinois.edu/RoboticSystems/Calibration.html)
